In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import * 
from pyspark.sql import DataFrame

In [0]:


bronze_posts_df = spark.read.table("default.posts")

def split_array(df:DataFrame):
    return df.withColumn("Split_Tags",filter(split(df["Tags"],r'\|'),lambda x: x != "")).drop("Tags")
                                                               
def rename_column(df:DataFrame):
    return df.withColumnRenamed("Id","PostId")

def convert_to_map(df:DataFrame):
    map_data = [
        (1, "Question"),
        (2, "Answer"),
        (3, "Orphaned tag wiki"),
        (4, "Tag wiki excerpt"),
        (5, "Tag wiki"),
        (6, "Moderator nomination"),
        (7, "Wiki placeholder"),
        (8, "Privilege wiki"),
        (9, "Article"),
        (10, "HelpArticle"),
        (12, "Collection"),
        (13, "ModeratorQuestionnaireResponse"),
        (14, "Announcement"),
        (15, "CollectiveDiscussion"),
        (17, "CollectiveCollection")
    ]
    map_schema = StructType([
        StructField("PostTypeId", IntegerType(),False),
        StructField("PostType", StringType(),False)
    ])
    map_df = spark.createDataFrame(map_data, schema=map_schema)

    return df.join(broadcast(map_df), df["PostTypeId"] == map_df["PostTypeId"],"left").drop(map_df["PostTypeId"])

staging_df = bronze_posts_df.transform(split_array).transform(rename_column).transform(convert_to_map)



In [0]:
bronze_posts_df.printSchema()

In [0]:
display(staging_df.limit(5))

In [0]:
from delta.tables import DeltaTable

def incremental_upsert(dest_table: str, df: DataFrame, unique_key: str, updated_at: str, full_refresh=False):
    if not spark.catalog.tableExists(dest_table) or full_refresh:
        (
            df
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(dest_table)
        )
    else:
        last_max = (
            spark.table(dest_table)
                .agg(max(updated_at).alias("max_ts"))
                .collect()[0]["max_ts"]
        )

        incr_df = df.filter(col(updated_at) > last_max)

        if not incr_df.isEmpty():
            delta_table = DeltaTable.forName(spark, dest_table)
            (
                delta_table.alias("t")
                    .merge(
                        source=incr_df.alias("s"),
                        condition=f"s.{unique_key} = t.{unique_key}"
                    )
                    .whenMatchedUpdateAll()
                    .whenNotMatchedInsertAll()
                    .execute()
            )

dest_table = "default.stg_posts"
incremental_upsert(dest_table, staging_df, "PostId", "CreationDate")
     


In [0]:
incremental_upsert(dest_table, staging_df.repartition(4), "PostId", "CreationDate", full_refresh=True)
     

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 4)
     

In [0]:
display(spark.table(dest_table).limit(5))